# Evaluate the frozen PS5 models
**Local Jupyter:** extract the submission ZIP, open this notebook in the extracted folder and place the two labeled test CSVs there. **Google Colab:** open this notebook and use the Files panel to upload the submission ZIP and both test CSVs.

Set the two filenames below, then choose **Run all**. Results appear in the notebook and in the `results` folder. On first use, setup may download Python and the pinned packages. Use a trusted copy of the submission bundle.

In [ ]:
from pathlib import Path

# Enter the actual filenames you uploaded, including the .csv extension, or supply full paths. Paths are relative to the notebook's working directory.
FRAUD_TEST = Path("fraud_test.csv")          # Header row; is_fraud is the final column
ACTIVITY_TEST = Path("activity_test.csv")   # No header; 328 sensors followed by activity name
SUBMISSION_ZIP = Path("PS5 Team Submission.zip")  # Used when the submission is still zipped
OUTPUT = Path("results").resolve()

## Locate the frozen system
The notebook uses the same standalone evaluator that the grader can run from the command line.

In [ ]:
import os
import shutil
import subprocess
import sys
import tempfile
import zipfile
from importlib import metadata

expected = [
    "evaluate.py", "requirements.txt",
    "best_models/fraud_best_random_forest_transaction_history.pkl",
    "best_models/activity_best_15_neighbor_sports_booster_ensemble.pkl",
]
for csv_path in (FRAUD_TEST, ACTIVITY_TEST):
    if not csv_path.is_file():
        available = ", ".join(sorted(p.name for p in Path.cwd().glob("*.csv"))) or "(none)"
        raise FileNotFoundError(f"Missing {csv_path}. CSVs in {Path.cwd()}: {available}. "
                                "Set the filename in the first code cell and rerun it.")
bundle = next((root for root in (Path.cwd(), Path.cwd() / "PS5 Submission")
               if all((root / name).is_file() for name in expected)), None)
if bundle is None:
    if not SUBMISSION_ZIP.is_file():
        raise FileNotFoundError("Upload the submission ZIP or start this notebook in the extracted folder.")
    scratch = tempfile.TemporaryDirectory()
    bundle = Path(scratch.name)
    with zipfile.ZipFile(SUBMISSION_ZIP) as archive:
        for name in expected:
            archive.extract(name, bundle)
print("Located both CSVs and frozen models.")

## Run and save predictions
Setup first checks the installed package versions. If they differ, uv prepares a separate Python 3.12 runtime for the evaluator. Initial setup needs internet access or a populated package cache. The notebook kernel can stay open throughout.

In [ ]:
def version_matches(line):
    package, wanted = line.split("==")
    try:
        return metadata.version(package) == wanted
    except metadata.PackageNotFoundError:
        return False

requirements = bundle / "requirements.txt"
pins = [s.strip() for s in requirements.read_text(encoding="utf-8-sig").splitlines()
        if s.strip() and not s.lstrip().startswith("#")]
environment = os.environ.copy()
mismatches = [pin for pin in pins if not version_matches(pin)]
print("Notebook Python:", sys.version.split()[0], "| Required evaluator Python: 3.12")
if mismatches:
    print("Missing or different package versions; evaluator requires:", ", ".join(mismatches))
if sys.version_info[:2] == (3, 12) and not mismatches:
    launch = [sys.executable]
    print("Using the matching installed Python and packages.", flush=True)
else:
    print("Preparing a separate runtime with the export versions. First setup can take several minutes.", flush=True)
    uv = shutil.which("uv")
    if not uv:
        bootstrap = tempfile.TemporaryDirectory()
        subprocess.run([sys.executable, "-m", "pip", "install", "--target", bootstrap.name,
                        "uv==0.13.9"], check=True)
        binary = "uv.exe" if os.name == "nt" else "uv"
        uv = str(next(path for path in Path(bootstrap.name).rglob(binary) if path.is_file()))
    for key in ("PYTHONPATH", "PYTHONHOME", "VIRTUAL_ENV"):
        environment.pop(key, None)
    launch = [uv, "run", "--no-project", "--isolated", "--python", "3.12",
              "--with-requirements", str(requirements.resolve()), "--", "python"]

args = [str(bundle / "evaluate.py"), "--fraud-test", str(FRAUD_TEST.resolve()),
        "--activity-test", str(ACTIVITY_TEST.resolve()), "--output-dir", str(OUTPUT)]
run = subprocess.run(launch + args, env=environment, capture_output=True, text=True)
print(run.stdout)
if run.returncode:
    print(run.stderr, file=sys.stderr, flush=True)
run.check_returncode()
# Keep a plain-text record of the same six metrics printed above.
(OUTPUT / "metrics.txt").write_text(run.stdout, encoding="utf-8")
print("Saved metrics.txt, fraud_predictions.csv and activity_predictions.csv in", OUTPUT)